# Grupo 1 | MCDI501 - Estadística Computacional para la Toma de Decisiones

## Integrantes
- Pablo Ignacio Balbontín Constenla @pabbalbontin-maker
- Melany Esmeralda Reyes Leiva @melanyreyesy
- Ingeborg Andrea Muñoz Carnot @dark452
- Mario Alejandro López Pulgar @malp2203

## Descripción del problema - Fase 4: Modelamiento Predictivo Integrado (Sumativa 3)

**Proyecto**: Predicción de la Deserción y el Éxito Académico de los Estudiantes

Este cuaderno integra los resultados de las Sumativas 1 y 2 en un modelo predictivo de
deserción estudiantil. El manejo de faltantes, la selección de variables, la lectura de la
colinealidad y la validación por bootstrap se fundamentan en los hallazgos de S1
(diagnóstico de calidad, correlaciones, pruebas de hipótesis) y S2 (correlaciones validadas
por bootstrap, pruebas confirmadas por permutación, observaciones influyentes por jackknife).

**Dataset:** *Predict Students' Dropout and Academic Success* (Realinho et al., 2022), UCI Repository.
$n = 4.424$ estudiantes, 37 variables.

**Estructura del cuaderno** (progresión S1 \$ \rightarrow \$ S2 \$ \rightarrow \$ S3 explícita en cada sección):

1. Manejo inteligente de datos faltantes
2. Clasificación mediante regresión logística
3. Análisis comparativo del impacto de la imputación
4. Síntesis integrada e informe final

**Semilla:** 42

## Configuración del entorno

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})
sns.set_style('whitegrid')

COLORS_TARGET = {'Graduate': '#2ca02c', 'Dropout': '#d62728', 'Enrolled': '#1f77b4'}
BOOT_COLOR = '#4c72b0'
PERM_COLOR = '#dd8452'
FIG_DIR = '../data/processed'

print('Entorno configurado.')

Entorno configurado.


In [2]:
def load_data(file_path):
    """Carga el dataset raw desde un archivo CSV.

    Parámetros
    ----------
    file_path : str
        Ruta del archivo CSV utilizado como entrada.

    Retorno
    -------
    pd.DataFrame
        Datos cargados en un DataFrame.

    Excepción
    ---------
    FileNotFoundError
        Si la ruta al archivo CSV no existe. Se muestra un mensaje de error.
    """
    try:
        df = pd.read_csv(file_path, sep=';')
    except FileNotFoundError:
        raise FileNotFoundError(
            f"No se encontro el archivo '{file_path}'. "
            "Verificar que el archivo CSV se encuentre en data/raw."
        )
    return df


df_full = load_data('../data/raw/predict_students_dropout_and_academic_success.csv')
df_full.columns = [c.strip() for c in df_full.columns]
print(f'Dataset cargado: {df_full.shape[0]:,} filas x {df_full.shape[1]} columnas')

Dataset cargado: 4,424 filas x 37 columnas


### Parámetros de referencia de las Sumativas 1 y 2

Los siguientes resultados, validados en la Sumativa 2 (sección 6, *"Preparación de insumos para la Sumativa 3"*), constituyen la base de las decisiones de modelamiento de esta fase.

In [3]:
# Correlaciones con el resultado académico validadas por bootstrap en S2 (IC 95%)
S2_CORR = {
    'Curricular units 2nd sem (grade)': {'lab': 'Nota 2do Semestre',    'r': 0.567, 'ci': (0.544, 0.589),   'clas': 'Robusta y relevante'},
    'Curricular units 1st sem (grade)': {'lab': 'Nota 1er Semestre',    'r': 0.485, 'ci': (0.461, 0.508),   'clas': 'Robusta y relevante'},
    'Age at enrollment':                {'lab': 'Edad al Matricularse', 'r': -0.243, 'ci': (-0.274, -0.213), 'clas': 'Robusta, relevancia moderada'},
    'Admission grade':                  {'lab': 'Nota de Admisión',     'r': 0.121, 'ci': (0.091, 0.150),   'clas': 'Robusta, relevancia baja'},
    'GDP':                              {'lab': 'PIB',                  'r': 0.044, 'ci': (0.016, 0.073),   'clas': 'Estable pero marginal'},
}

# Pruebas de hipótesis validadas por permutación en S2
S2_TESTS = {
    'Prueba 1': 'Nota de Admisión difiere entre Dropout y Graduate '
                '(Welch p=2.6e-14; permutación p=0.0001; Mann-Whitney p=3.3e-15)',
    'Prueba 2': 'Beca asociada al resultado académico '
                '(chi2 p=9.6e-90; permutación p=0.0001; V de Cramer=0.304)',
}

# Observaciones influyentes identificadas por jackknife en S2
S2_JACKKNIFE = ('Notas de admisión en el techo de escala (190) y estudiantes de 60-70 años; '
                'ningún caso altera las conclusiones al excluirlo.')

print('Referencias S1/S2 cargadas:')
for k, v in S2_CORR.items():
    print(f"  {v['lab']:<22} r={v['r']:+.3f}  IC=[{v['ci'][0]:+.3f}, {v['ci'][1]:+.3f}]  ({v['clas']})")
for k, v in S2_TESTS.items():
    print(f'  {k}: {v}')

Referencias S1/S2 cargadas:
  Nota 2do Semestre      r=+0.567  IC=[+0.544, +0.589]  (Robusta y relevante)
  Nota 1er Semestre      r=+0.485  IC=[+0.461, +0.508]  (Robusta y relevante)
  Edad al Matricularse   r=-0.243  IC=[-0.274, -0.213]  (Robusta, relevancia moderada)
  Nota de Admisión       r=+0.121  IC=[+0.091, +0.150]  (Robusta, relevancia baja)
  PIB                    r=+0.044  IC=[+0.016, +0.073]  (Estable pero marginal)
  Prueba 1: Nota de Admisión difiere entre Dropout y Graduate (Welch p=2.6e-14; permutación p=0.0001; Mann-Whitney p=3.3e-15)
  Prueba 2: Beca asociada al resultado académico (chi2 p=9.6e-90; permutación p=0.0001; V de Cramer=0.304)


### Definición del problema de clasificación

La variable objetivo original tiene tres clases (Graduate $49{,}9\%$ , Dropout $32{,}1\%$, Enrolled
$17{,}9\%$). Para el modelamiento se adopta el problema **binario Dropout (1) vs. Graduate (0)**,
excluyendo a los estudiantes Enrolled, por tres razones:

1. *Enrolled* no es un resultado terminal: son estudiantes aún matriculados al cierre del
   período, cuyo desenlace (graduarse o desertar) todavía no se observa, incluirlos como
   clase contaminaría el aprendizaje con etiquetas censuradas.
2. Toda la evidencia inferencial de S1 y S2 (pruebas de Welch, permutación y Mann-Whitney)
   se construyó comparando precisamente Dropout vs. Graduate, de modo que el modelo hereda
   directamente esas señales validadas.
3. La decisión de negocio que motiva el proyecto (intervención temprana de retención)
   requiere distinguir quién abandonará de quién se graduará.

In [4]:
df = df_full[df_full['Target'].isin(['Dropout', 'Graduate'])].copy()
df['y'] = (df['Target'] == 'Dropout').astype(int)
print(f'Subconjunto de modelamiento: {len(df):,} estudiantes')
print(f'  Graduate (0): {(df.y == 0).sum():,} ({(df.y == 0).mean() * 100:.1f}%)')
print(f'  Dropout  (1): {(df.y == 1).sum():,} ({df.y.mean() * 100:.1f}%)')
print('Desbalance moderado: no requiere ponderación de clases (a diferencia del caso')

Subconjunto de modelamiento: 3,630 estudiantes
  Graduate (0): 2,209 (60.9%)
  Dropout  (1): 1,421 (39.1%)
Desbalance moderado: no requiere ponderación de clases (a diferencia del caso
